# 🟢 新手模式 | Beginner Mode

**不用写代码。** 拖动滑杆、点“运行实验”，看曲线和回放怎么变，再回答一个小问题，就完成一次探索。

**三步操作：**
1. 点菜单栏 Run ▶▶ Run All Cells（全部运行一次，等默认结果出来）；
2. 拖动下面的滑杆改一个关键参数；
3. 点 **▶ 运行实验**，看对比图和一句话解读。

**诚实说明：** 你看到的数字全部来自真实引擎（不是演示动画），和完整模式是同一口径。
新手模式只写 `labs/data/lite/` 下的完成记录，**不计入教师成绩册**；想拿实验成绩，请接着做对应的完整模式。

# E3 新手模式：跟车跟多远最合适？


## 直觉问题

MPC 跟驰控制器每 0.2 秒向前看 10 步、算一次最优加速度。它的目标是保持一个“期望间距”：

> 期望间距 = 车速 × 跟驰时距 + 2 米

- **时距大**：跟得远，安全，但浪费道路、容易被加塞。
- **时距小**：跟得紧，道路利用率高，但一点扰动就可能撞上。

本实验只让你调一个数：**跟驰时距**（0.6–2.0 秒）。注意：这里是纯运动学验证（MPC 是真求解，车辆是运动学模型），
真 esmini 闭环在完整模式里。场景里前车会在第 3 秒突然减速到 10 m/s、持续 2 秒——
没有这次扰动，理想匀速跟驰的误差恒为 0，滑杆拉了也看不出区别。

**你的任务在最后**：找到误差最小又不撞的时距。


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
sys.dont_write_bytecode = True

# 新手驱动：滑杆 -> 真实引擎 -> 对比图（引擎本身不在本本里，全部复用完整模式的同一批代码）
from common.lite_drivers import (
    plot_curves, plot_bars,
)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

LAST = {}  # 最近一次运行的结果（供“保存心得”格子使用）
print("✓ 新手驱动就绪：下面拖动滑杆、点运行即可")


In [ ]:
from common.lite_drivers import run_e3_lite, e3_baseline, interpret_e3, save_lite_record, render_lite_metric_card

hw = widgets.FloatSlider(
    value=1.0, min=0.6, max=2.0, step=0.1,
    description="跟驰时距 (s)：", style={"description_width": "120px"},
    layout=widgets.Layout(width="70%"),
)
run_btn = widgets.Button(description="▶ 运行实验", button_style="success")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output(wait=True)
        print(f"正在跑真 MPC 求解（时距 {hw.value:.1f}s，纯运动学验证 10 秒）……请稍等")
        base = e3_baseline()
        res = run_e3_lite(hw.value)
        out.clear_output(wait=True)
        
        display(plot_curves(
            [(base["times"], base["gap_errs"], "基线 1.0s", "-"),
             (res["times"], res["gap_errs"], f"你的 {hw.value:.1f}s", "-")],
            "时间 (s)", "期望间距跟踪误差 (m)", "MPC 跟驰间距误差：基线 vs 你的调参"))
        display(plot_curves(
            [(res["times"], res["accels"], f"你的 {hw.value:.1f}s 加速度", "-")],
            "时间 (s)", "控制加速度 (m/s²)", "自车加速度响应 (舒适性视角)"))
        
        badge_text = "🟢 安全零碰撞跟踪" if not res["collision"] else "🔴 跟驰间距过近碰撞"
        badge_color = "green" if not res["collision"] else "red"
        metrics_list = [
            ("最大间距误差", f"{res['max_err']} m"),
            ("基线间距误差", f"{base['max_err']} m"),
            ("平均求解耗时", f"{res['avg_solve_ms']} ms"),
            ("碰撞状态", "正常" if not res["collision"] else "碰撞"),
        ]
        display(render_lite_metric_card("E3 MPC 跟驰", badge_text, badge_color, metrics_list, interpret_e3(res)))
        
        save_lite_record("E3", {"headway": hw.value},
                         {"max_err": res["max_err"],
                          "collision": res["collision"]})
        LAST.clear()
        LAST.update({"exp": "E3", "params": {"headway": hw.value},
                     "metrics": {"max_err": res["max_err"],
                                 "collision": res["collision"]}})

run_btn.on_click(on_run)
display(hw, run_btn, out)
on_run(None)

## 🎯 找区间小任务

拖动滑杆，回答下面两个问题（写在下一个格子的心得里）：

1. 时距从 0.6 调到 2.0，最大间距误差是怎么变的？有没有一个“误差最小”的区间？
2. 看加速度曲线：时距很小时，加速度是不是更抖（舒适性更差）？时距对安全和舒适分别意味着什么？


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from common.lite_drivers import save_lite_record

note = widgets.Textarea(
    value="",
    placeholder="写一句话：你找到了什么规律？（例如：阈值越大，刹车越早，但……）",
    description="我的心得：",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_btn = widgets.Button(description="💾 保存心得与打卡记录", button_style="info")
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        save_out.clear_output(wait=True)
        if not LAST:
            print("⚠️ 请先运行一次实验（上一个格子的“运行实验”按钮），再保存。")
            return
        ans = note.value.strip()
        if not ans:
            print("⚠️ 请在上方输入框写下一句寻优心得后再打卡保存！")
            return
        path = save_lite_record(LAST["exp"], LAST["params"], LAST["metrics"], note=ans)
        display(HTML("<span style='color: #16a34a; font-weight: bold;'>🎉 恭喜！新手探索打卡成功，已同步归档入预习数据库！</span><br/><span style='color: #64748b; font-size: 12px;'>注：新手模式完成记录独立于 29+1 课程成绩册，不影响期末考核。</span>"))

save_btn.on_click(_on_save)
display(note, save_btn, save_out)

---

## 下一步 | Next

- 完成上面的“找区间”小任务后，把你的心得保存下来（上一个格子）。
- 想拿实验成绩、学完整算法：去做对应的完整模式（同目录下不带 `_lite` 的 notebook）。
- 新手模式的意义是先建立直觉：参数变了、结果往哪个方向变、变化有多大。

完整模式：`E3_mpc_follow_pro.ipynb`
